# 00_recipe

Pick the training recipe (Track Paper P1+P2) and run the leak test (P0). RUN THIS FIRST.

Settings (right sidebar): **Internet = On**, Accelerator = **None** (CPU). Then *Save Version -> Save & Run All (Commit)* so it runs in the background.

In [ ]:
import os, subprocess, sys
REPO = "https://github.com/SonponYe/SECIoHT-FL-based-IDS-2.0.git"
os.chdir("/kaggle/working")
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "repo"], check=True)
else:
    subprocess.run(["git", "-C", "repo", "pull"], check=True)
os.chdir("/kaggle/working/repo/our_work")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opacus", "imbalanced-learn", "scipy"], check=True)
print("CPU cores:", os.cpu_count())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
import subprocess, sys, time

def run_parallel(module, stage, protocol, seeds=(0, 1, 2), extra=()):
    """One process per seed (1 thread each) so the 4 cores stay busy. Results append to results.jsonl."""
    t0, procs = time.time(), []
    for s in seeds:
        cmd = [sys.executable, "-m", module, "--stage", stage, "--protocol", protocol,
               "--seeds", str(s), "--threads", "1", *extra]
        log = open(f"log_{module.split('.')[-1]}_{stage}_{protocol}_s{s}.log", "w")
        procs.append((s, subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)))
    for s, p in procs:
        rc = p.wait()
        print(f"{module} {stage} {protocol} seed {s}: exit {rc}")
        assert rc == 0, f"failed - see log_{module.split('.')[-1]}_{stage}_{protocol}_s{s}.log"
    print(f"{stage}/{protocol} done in {(time.time()-t0)/60:.1f} min")

In [ ]:
run_parallel("track_paper.run_paper", "optim", "both")

In [ ]:
subprocess.run([sys.executable, "-m", "track_paper.run_paper", "--stage", "threshold", "--protocol", "both"], check=True)
print(open("shared/best_recipe.json").read())

In [ ]:
run_parallel("track_paper.run_paper", "reproduce", "both")

In [ ]:
import zipfile, glob
out = "/kaggle/working/results_00_recipe.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for pat in ["track_paper/results/*", "track_moe/results/*", "shared/best_recipe.json", "log_*.log"]:
        for f in glob.glob(pat):
            if os.path.isfile(f):
                z.write(f)
print("saved", out, os.path.getsize(out) // 1024, "KB")